In [2]:
# Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [3]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [4]:
reference_date = F.lit("2026-07-05").cast("date")
lookback_days = 30
year_days = 365
prior_year_weighting_factor = 0.75
lift_threshold = 1


TABLES = {
    "read": {
        "product_catalog": "marketingdata_prod.warehouse.product_catalog",
        "product_catalog_history": "marketingdata_prod.warehouse.product_catalog_history",
        "next_ads_sort_order": "marketingdata_prod.warehouse.next_ads_sort_order",
        "bq_views_next_uk": "marketingdata_prod.warehouse.bq_views_next_uk",
        "bq_views_next_uk_app": "marketingdata_prod.warehouse.bq_views_next_uk_app",
        "bq_atbs_next_uk": "marketingdata_prod.warehouse.bq_atbs_next_uk",
        "bq_atbs_next_uk_app": "marketingdata_prod.warehouse.bq_atbs_next_uk_app",
    },
}

In [5]:
SORT_ORDER_LATEST = TABLES["read"]["next_ads_sort_order"]
VIEWS = TABLES["read"]["bq_views_next_uk"]
VIEWS_APP = TABLES["read"]["bq_views_next_uk_app"]
ATBS = TABLES["read"]["bq_atbs_next_uk"]
ATBS_APP = TABLES["read"]["bq_atbs_next_uk_app"]
PRODUCT_CATALOG = TABLES["read"]["product_catalog"]
PRODUCT_CATALOG_HISTORY = TABLES["read"]["product_catalog_history"]

In [6]:
## Unique CatID by PID
# Note: some may have multiple associated due to differing information on different items in the range - we have taken the most common catid

product_cat_ids_current = (
    spark.table(PRODUCT_CATALOG)
    .withColumn(
        "catid",
        F.concat_ws(
            "_",
            F.when(
                F.col("department") == "childrenswear", F.col("next_gender")
            ).otherwise(F.col("department")),
            F.col("brand"),
            F.col("next_category"),
        ),
    )
    .groupBy(F.col("pid"), F.col("catid"))
    .agg(F.count("*").alias("number_items"))
    .withColumn(
        "pid_max",
        F.row_number().over(
            Window.partitionBy("pid").orderBy(
                F.desc(F.col("number_items")), F.desc(F.col("catid"))
            )
        ),
    )
    .filter(F.col("pid_max") == 1)
    .select(F.col("pid").alias("itemno"), F.col("catid"))
)

prod_cat_history = (
    spark.table(PRODUCT_CATALOG_HISTORY)
    .filter(F.col("rundate") >= F.date_sub(reference_date, year_days))
    .withColumn(
        "catid",
        F.concat_ws(
            "_",
            F.when(
                F.col("department") == "childrenswear", F.col("next_gender")
            ).otherwise(F.col("department")),
            F.col("brand"),
            F.col("next_category"),
        ),
    )
    .groupBy(F.col("pid"), F.col("catid"))
    .agg(F.count("*").alias("number_items"))
    .withColumn(
        "pid_max",
        F.row_number().over(
            Window.partitionBy("pid").orderBy(
                F.desc(F.col("number_items")), F.desc(F.col("catid"))
            )
        ),
    )
    .filter(F.col("pid_max") == 1)
    .select(F.col("pid").alias("itemno"), F.col("catid"))
    .join(product_cat_ids_current, how="left_anti", on="itemno")
)
product_cat_ids = product_cat_ids_current.union(prod_cat_history)

In [7]:
# Unique Advert Items
## Some of the items use sku_id/ some use pid from the sort order data ; joining to account for both here

sort_order = spark.table(SORT_ORDER_LATEST)
product_cat = spark.table(PRODUCT_CATALOG)
advert_items = (
    sort_order.filter(
        (F.col("rundate") == reference_date) & (F.col("Status") == "Active")
    )
    .join(
        product_cat, on=(sort_order["items"] == product_cat["pid"]), how="left"
    )
    .withColumnRenamed("pid", "pid_")
    .select("items", "UniqueAdID", sort_order["rundate"], "pid_")
    .join(
        product_cat,
        on=(sort_order["items"] == product_cat["sku_id"]),
        how="left",
    )
    .withColumn(
        "itemno", F.coalesce(F.col("pid_"), F.col("pid"), F.col("items"))
    )
    .join(product_cat_ids, on="itemno", how="left")
    .select("itemno", "UniqueAdID", sort_order["rundate"], "catid")
    .distinct()
)

advert_catids = advert_items.select(
    "UniqueAdID", "rundate", "catid"
).distinct()

In [8]:
# Determine Ad profile similarity

ad_items_array = advert_items.groupBy(
    F.col("UniqueAdID"), F.col("rundate")
).agg(
    F.array_distinct(F.collect_list("itemno")).alias("items_list"),
    F.countDistinct("itemno").alias("itemcount"),
)

ad_items_overlap = (
    ad_items_array.alias("a")
    .crossJoin(ad_items_array.alias("b"))
    .withColumn(
        "intersect_array",
        F.array_intersect(F.col("a.items_list"), F.col("b.items_list")),
    )
    .withColumn("intersection_count", F.size(F.col("intersect_array")))
    .withColumn(
        "overlap_proportion",
        F.col("intersection_count") / F.col("a.itemcount"),
    )
    .select(
        F.col("a.UniqueAdID"),
        F.col("b.UniqueAdID").alias("TargetUniqueAdID"),
        F.col("a.itemcount"),
        F.col("b.itemcount").alias("target_itemcount"),
        "intersection_count",
        "overlap_proportion",
    )
)

In [ ]:
# display(ad_items_overlap.filter(F.col("UniqueAdID")=='P151_C1285_Brands_Girls _JojoMamanBebe_PLP_Girls').orderBy(F.desc(F.col("overlap_proportion"))))

In [ ]:
# advert_items.count()
# advert_items.cache()

# advert_catids.count()
# advert_catids.cache()

In [9]:
# Recent history Views & Add to Basket datasets:

# Views
web_views = (
    spark.table(VIEWS)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, lookback_days), reference_date
            )
        )
        & (F.col("ViewTimeSpentSecs") > 0)
        & (F.col("EventType").ilike("pdp_view"))
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .groupBy((F.col("itemno")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)


app_views = (
    spark.table(VIEWS_APP)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, lookback_days), reference_date
            )
        )
        & (F.col("ViewTimeSpentSecs") > 0)
        & (F.col("ScreenName") == "PDP")
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .groupBy((F.col("itemno")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)

all_ad_item_views = web_views.union(app_views)

# Add to basket
web_atbs = (
    spark.table(ATBS)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, lookback_days), reference_date
            )
        )
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .groupBy((F.col("itemno")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)

app_atbs = (
    spark.table(ATBS_APP)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, lookback_days), reference_date
            )
        )
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .groupBy((F.col("itemno")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)

all_ad_item_atbs = web_atbs.union(app_atbs)

In [10]:
# All view:items combined (associated to advert):
associated_view_basket_items = (
    all_ad_item_views.join(
        all_ad_item_atbs,
        how="inner",
        on=(
            (
                all_ad_item_atbs["UniqueVisitID"]
                == all_ad_item_views["UniqueVisitID"]
            )
            & (all_ad_item_atbs["Timestamp"] > all_ad_item_views["Timestamp"])
        ),
    )
).select(
    all_ad_item_views["itemno"].alias("viewitem"),
    all_ad_item_atbs["itemno"].alias("atbitem"),
    all_ad_item_views["date"],
    all_ad_item_views["UniqueVisitID"],
)

# Roll up to advert level & day
basket_advert_items = (
    associated_view_basket_items.join(
        F.broadcast(advert_items),
        on=(
            associated_view_basket_items["viewitem"] == advert_items["itemno"]
        ),
        how="inner",
    )
    .select("date", "UniqueVisitID", "UniqueAdID", "atbitem")
    .withColumnRenamed("UniqueAdID", "ViewUniqueAdvertID")
)
advert_item_associations = (
    basket_advert_items.join(
        F.broadcast(advert_items),
        on=associated_view_basket_items["atbitem"] == advert_items["itemno"],
        how="inner",
    )
    .select("date", "UniqueVisitID", "ViewUniqueAdvertID", "UniqueAdID")
    .withColumnRenamed("UniqueAdID", "AtbUniqueAdvertID")
    .distinct()
)

In [11]:
## Aggregations of data for recent history - AdvertID Level
total_sessions = (
    associated_view_basket_items.select("UniqueVisitID").distinct().count()
)
number_views = advert_item_associations.groupBy("ViewUniqueAdvertID").agg(
    F.countDistinct("UniqueVisitID").alias("number_views")
)
number_atbs = advert_item_associations.groupBy("AtbUniqueAdvertID").agg(
    F.countDistinct("UniqueVisitID").alias("number_atbs")
)
number_views_atbs = advert_item_associations.groupBy(
    "ViewUniqueAdvertID", "AtbUniqueAdvertID"
).agg(F.countDistinct("UniqueVisitID").alias("number_views_atbs"))

In [12]:
# ## Build association metrics
association = (
    number_views_atbs.join(number_views, how="left", on="ViewUniqueAdvertID")
    .join(number_atbs, how="left", on="AtbUniqueAdvertID")
    .withColumn("support_views", (F.col("number_views") / total_sessions))
    .withColumn("support_atbs", (F.col("number_atbs") / total_sessions))
    .withColumn(
        "support_views_atbs", (F.col("number_views_atbs") / total_sessions)
    )
    .withColumn(
        "cosine_similarity",
        (
            F.col("number_views_atbs")
            / (F.sqrt(F.col("number_views")) * F.sqrt(F.col("number_atbs")))
        ),
    )
    .withColumn(
        "lift",
        (
            F.col("support_views_atbs")
            / (F.col("support_views") * F.col("support_atbs"))
        ),
    )
    # .withColumn("lift_adjusted",((F.col("support_views_atbs")/ (F.col("support_views")* F.col("support_atbs")))* F.power(F.col("support_atb"), F.lit(0.25))))
)

In [13]:
## CatId level of 11 months prior ( seasonality 'forward look')
## has to be done at catid level - product lifecycle is too short for direct item relationship

# Views dataset
web_views_prior_year = (
    spark.table(VIEWS)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, year_days),
                F.date_sub(reference_date, year_days - lookback_days),
            )
        )
        & (F.col("ViewTimeSpentSecs") > 0)
        & (F.col("EventType").ilike("pdp_view"))
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .join(product_cat_ids, how="inner", on="itemno")
    .groupBy((F.col("catid")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)

app_views_prior_year = (
    spark.table(VIEWS_APP)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, year_days),
                F.date_sub(reference_date, year_days - lookback_days),
            )
        )
        & (F.col("ViewTimeSpentSecs") > 0)
        & (F.col("ScreenName") == "PDP")
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .join(product_cat_ids, how="inner", on="itemno")
    .groupBy((F.col("catid")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)


all_ad_item_views_prior_year = web_views_prior_year.union(app_views_prior_year)


# Add to basket dataset
web_atbs_prior_year = (
    spark.table(ATBS)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, year_days),
                F.date_sub(reference_date, year_days - lookback_days),
            )
        )
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .join(product_cat_ids, how="inner", on="itemno")
    .groupBy((F.col("catid")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)


app_atbs_prior_year = (
    spark.table(ATBS_APP)
    .filter(
        (
            F.col("date").between(
                F.date_sub(reference_date, year_days),
                F.date_sub(reference_date, year_days - lookback_days),
            )
        )
    )
    .withColumnRenamed("ProductSKU", "itemno")
    .join(product_cat_ids, how="inner", on="itemno")
    .groupBy((F.col("catid")), F.col("date"), F.col("UniqueVisitID"))
    .agg(F.min(F.col("Timestamp")).alias("Timestamp"))
)

all_ad_item_atbs_prior_year = web_atbs_prior_year.union(app_atbs_prior_year)

In [14]:
# All view:items combined (associated to advert):
associated_view_basket_items_prior_year = (
    all_ad_item_views_prior_year.join(
        all_ad_item_atbs_prior_year,
        how="inner",
        on=(
            (
                all_ad_item_atbs_prior_year["UniqueVisitID"]
                == all_ad_item_views_prior_year["UniqueVisitID"]
            )
            & (
                all_ad_item_atbs_prior_year["Timestamp"]
                > all_ad_item_views_prior_year["Timestamp"]
            )
        ),
    )
).select(
    all_ad_item_views_prior_year["catid"].alias("viewcatid"),
    all_ad_item_atbs_prior_year["catid"].alias("atbcatid"),
    all_ad_item_views_prior_year["date"],
    all_ad_item_views_prior_year["UniqueVisitID"],
)
# Roll up to advert level & day
basket_advert_items_prior_year = (
    associated_view_basket_items_prior_year.join(
        F.broadcast(advert_catids),
        on=(
            associated_view_basket_items_prior_year["viewcatid"]
            == advert_items["catid"]
        ),
        how="inner",
    )
    .select("date", "UniqueVisitID", "UniqueAdID", "atbcatid")
    .withColumnRenamed("UniqueAdID", "ViewUniqueAdvertID")
)

advert_item_associations_prior_year = (
    basket_advert_items_prior_year.join(
        F.broadcast(advert_catids),
        on=(
            associated_view_basket_items_prior_year["atbcatid"]
            == advert_items["catid"]
        ),
        how="inner",
    )
    .select("date", "UniqueVisitID", "ViewUniqueAdvertID", "UniqueAdID")
    .withColumnRenamed("UniqueAdID", "AtbUniqueAdvertID")
    .distinct()
)

In [15]:
## Aggregations of data
# total_sessions_prior_year=associated_view_basket_items_prior_year.select("UniqueVisitID").distinct().count()
number_views_prior_year = advert_item_associations_prior_year.groupBy(
    "ViewUniqueAdvertID"
).agg(F.countDistinct("UniqueVisitID").alias("number_views"))
number_atbs_prior_year = advert_item_associations_prior_year.groupBy(
    "AtbUniqueAdvertID"
).agg(F.countDistinct("UniqueVisitID").alias("number_atbs"))
number_views_atbs_prior_year = advert_item_associations_prior_year.groupBy(
    "ViewUniqueAdvertID", "AtbUniqueAdvertID"
).agg(F.countDistinct("UniqueVisitID").alias("number_views_atbs"))

In [ ]:
# ## Build association metrics
# association_prior_year=(number_views_atbs_prior_year.join(number_views_prior_year, how="left", on="ViewUniqueAdvertID").join(number_atbs_prior_year, how="left", on="AtbUniqueAdvertID")
# .withColumn("support_views",(F.col("number_views")/total_sessions_prior_year))
# .withColumn("support_atbs",(F.col("number_atbs")/total_sessions_prior_year))
# .withColumn("support_views_atbs",(F.col("number_views_atbs")/total_sessions_prior_year))
# .withColumn("cosine_similarity",(F.col("number_views_atbs")/(F.sqrt(F.col("number_views")) * F.sqrt(F.col("number_atbs")))))
# .withColumn("lift",(F.col("support_views_atbs")/ (F.col("support_views")* F.col("support_atbs"))))
# #.withColumn("lift_adjusted",((F.col("support_views_atbs")/ (F.col("support_views")* F.col("support_atbs")))* F.power(F.col("support_atb"), F.lit(0.25))))
# )

In [16]:
total_sessions_ = (
    associated_view_basket_items_prior_year.select("UniqueVisitID")
    .distinct()
    .count()
    * prior_year_weighting_factor
) + associated_view_basket_items.select("UniqueVisitID").distinct().count()

In [17]:
number_views_combined = (
    number_views.join(
        number_views_prior_year, on="ViewUniqueAdvertID", how="left"
    ).withColumn(
        "number_views_",
        number_views["number_views"]
        + (
            number_views_prior_year["number_views"]
            * F.lit(prior_year_weighting_factor)
        ),
    )
).select(number_views["ViewUniqueAdvertID"], "number_views_")

number_atbs_combined = (
    number_atbs.join(
        number_atbs_prior_year, on="AtbUniqueAdvertID", how="left"
    ).withColumn(
        "number_atbs_",
        number_atbs["number_atbs"]
        + (
            number_atbs_prior_year["number_atbs"]
            * F.lit(prior_year_weighting_factor)
        ),
    )
).select(number_atbs["AtbUniqueAdvertID"], "number_atbs_")

number_views_atbs_combined = (
    number_views_atbs.join(
        number_views_atbs_prior_year,
        on=(
            (
                number_views_atbs["ViewUniqueAdvertID"]
                == number_views_atbs_prior_year["ViewUniqueAdvertID"]
            )
            & (
                number_views_atbs["AtbUniqueAdvertID"]
                == number_views_atbs_prior_year["AtbUniqueAdvertID"]
            )
        ),
        how="left",
    ).withColumn(
        "number_views_atbs_",
        number_views_atbs["number_views_atbs"]
        + (
            number_views_atbs_prior_year["number_views_atbs"]
            * F.lit(prior_year_weighting_factor)
        ),
    )
).select(
    number_views_atbs["AtbUniqueAdvertID"],
    number_views_atbs["ViewUniqueAdvertID"],
    "number_views_atbs_",
)

In [19]:
## Build association metrics
association_ = (
    number_views_atbs_combined.alias("base")
    .join(
        number_views_combined.alias("views"),
        how="left",
        on="ViewUniqueAdvertID",
    )
    .join(
        number_atbs_combined.alias("atb"), how="left", on="AtbUniqueAdvertID"
    )
    .join(
        F.broadcast(ad_items_overlap).alias("overlap"),
        on=(
            ("base.ViewUniqueAdvertID" == "overlap.UniqueAdID")
            & ("base.AtbUniqueAdvertID" == "overlap.TargetUniqueAdID")
        ),
        how="left",
    )
    .withColumn("support_views", (F.col("number_views_") / total_sessions_))
    .withColumn("support_atbs", (F.col("number_atbs_") / total_sessions_))
    .withColumn(
        "support_views_atbs", (F.col("number_views_atbs_") / total_sessions_)
    )
    .withColumn(
        "cosine_similarity",
        (
            F.col("number_views_atbs_")
            / (F.sqrt(F.col("number_views_")) * F.sqrt(F.col("number_atbs_")))
        ),
    )
    .withColumn(
        "lift",
        (
            F.col("support_views_atbs")
            / (F.col("support_views") * F.col("support_atbs"))
        ),
    )
    .withColumn(
        "lift_adjusted",
        (
            (
                F.col("support_views_atbs")
                / (F.col("support_views") * F.col("support_atbs"))
            )
            * F.power(F.col("support_atbs"), F.lit(0.25))
        )
        / (F.lit(1) + F.coalesce(F.col("overlap_proportion"), F.lit(0))),
    )
    .filter(F.col("lift_adjusted") > F.lit(lift_threshold))
    .select(
        F.col("base.ViewUniqueAdvertID"),
        F.col("base.AtbUniqueAdvertID"),
        F.col("base.number_views_atbs_").alias("number_views_atbs"),
        F.col("views.number_views_").alias("number_views"),
        F.col("atbs.number_atbs_").alias("number_atbs"),
        F.col("support_views"),
        F.col("support_atbs"),
        F.col("support_views_atbs"),
        F.col("cosine_similarity"),
        F.col("lift"),
        F.col("lift_adjusted"),
    )
)

In [ ]:
display(
    association_.filter(
        F.col("ViewUniqueAdvertID")
        == "P151_C1285_Brands_Girls _JojoMamanBebe_PLP_Girls"
    ).orderBy(F.desc(F.col("lift")))
)

In [20]:
display(
    association_.filter(
        F.col("ViewUniqueAdvertID")
        == "P151_C1285_Brands_Girls _JojoMamanBebe_PLP_Girls"
    ).orderBy(F.desc(F.col("lift_adjusted")))
)

: 

In [ ]:
display(
    association.filter(
        F.col("ViewUniqueAdvertID")
        == "P151_C1285_Brands_Girls _JojoMamanBebe_PLP_Girls"
    ).orderBy(F.desc(F.col("lift")))
)

# association_.filter(F.col("ViewUniqueAdvertID")=="P151_C1285_Brands_Girls _JojoMamanBebe_PLP_Girls")

In [ ]:
# Ranking of ads & association cut off
## Adjusted_Lift > 1


## Validation check  & addition of columns for sensible logging to feature store e.g. runid & advert location
# build this out to replicate V2 structure

